In [ ]:
from pathlib import Path

import joblib
import pandas as pd
import torch
from torch import nn

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

encoder = joblib.load(
    project_root / "data/processed/lab06/category_encoder.joblib"
)

columns = encoder.feature_names_in_.tolist()

examples = pd.DataFrame(
    [
        ["US Dollar", "US Dollar", "ACH"],
        ["Euro", "US Dollar", "ACH"],
    ],
    columns=columns,
)

category_ids = torch.from_numpy(
    encoder.transform(examples) + 1
)

examples

,payment_currency,receiving_currency,payment_format
0,US Dollar,US Dollar,ACH
1,Euro,US Dollar,ACH


In [2]:
torch.set_num_threads(2)
torch.manual_seed(42)

embedding_tables = nn.ModuleList([
    nn.Embedding(
        num_embeddings=len(categories) + 1,
        embedding_dim=8,
        padding_idx=0,
    )
    for categories in encoder.categories_
])

attention = nn.MultiheadAttention(
    embed_dim=8,
    num_heads=2,
    dropout=0.0,
    batch_first=True,
)

attention.eval()

with torch.inference_mode():
    tokens = torch.stack(
        [
            table(category_ids[:, column])
            for column, table in enumerate(embedding_tables)
        ],
        dim=1,
    )

    contextual_tokens, attention_weights = attention(
        tokens,
        tokens,
        tokens,
        need_weights=True,
        average_attn_weights=False,
    )

print("Input tokens:", tokens.shape)
print("Output tokens:", contextual_tokens.shape)
print("Attention weights:", attention_weights.shape)

Input tokens: torch.Size([2, 3, 8])
Output tokens: torch.Size([2, 3, 8])
Attention weights: torch.Size([2, 2, 3, 3])


In [3]:
attention_table = pd.DataFrame(
    attention_weights[0, 0].numpy(),
    index=columns,
    columns=columns,
)

attention_table.index.name = "query feature"
attention_table.columns.name = "key feature"

attention_table.round(3)

key feature,payment_currency,receiving_currency,payment_format
query feature,,,
payment_currency,0.329,0.289,0.382
receiving_currency,0.367,0.360,0.273
payment_format,0.136,0.737,0.127


In [4]:
row_sums = attention_weights.sum(dim=-1)

torch.testing.assert_close(
    row_sums,
    torch.ones_like(row_sums),
)

print("Every attention row sums to one.")

Every attention row sums to one.


In [5]:
payment_column = columns.index("payment_format")

raw_change = torch.linalg.vector_norm(
    tokens[1, payment_column] - tokens[0, payment_column]
)

contextual_change = torch.linalg.vector_norm(
    contextual_tokens[1, payment_column]
    - contextual_tokens[0, payment_column]
)

print(f"Raw ACH vector change: {raw_change.item():.4f}")
print(
    f"ACH vector change after attention: "
    f"{contextual_change.item():.4f}"
)

Raw ACH vector change: 0.0000
ACH vector change after attention: 0.7122


In [8]:
import sys
from time import perf_counter

import numpy as np
from sklearn.metrics import average_precision_score
from torch.utils.data import DataLoader, TensorDataset

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.embeddings import predict_embedding_scores
from src.neural_training import train_one_epoch
from src.tabular_transformer import TabularTransformer

torch.set_num_threads(2)

previous_cache = project_root / "data/processed/lab05"
category_cache = project_root / "data/processed/lab06"
cache_dir = project_root / "data/processed/lab07"
cache_dir.mkdir(parents=True, exist_ok=True)

preprocessor = joblib.load(previous_cache / "preprocessor.joblib")
n_numeric = preprocessor.output_indices_["categorical"].start
assert n_numeric == 8

vocabulary_sizes = tuple(len(c) + 1 for c in encoder.categories_)


def load_inputs(split):
    previous = np.load(
        previous_cache / f"X_{split}.npy",
        mmap_mode="r",
    )
    numeric = np.array(
        previous[:, :n_numeric],
        dtype=np.float32,
        order="C",
        copy=True,
    )
    categories = np.load(category_cache / f"X_cat_{split}.npy")
    labels = np.load(previous_cache / f"y_{split}.npy")

    assert len(numeric) == len(categories) == len(labels)

    return (
        torch.from_numpy(numeric),
        torch.from_numpy(categories),
        torch.from_numpy(labels),
    )


train_numeric, train_categories, y_train = load_inputs("train")
val_numeric, val_categories, y_validation = load_inputs("validation")

In [9]:
reports_dir = project_root / "reports"
models_dir = project_root / "models"
reports_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

epochs = 5
training_history = []

for model_name, use_transformer in [
    ("embedding_control", False),
    ("transformer", True),
]:
    # Reset before EACH model so shared weights start identically.
    torch.manual_seed(42)

    model_config = {
        "n_numeric": n_numeric,
        "vocabulary_sizes": vocabulary_sizes,
        "positive_rate": y_train.mean().item(),
        "token_dim": 8,
        "num_heads": 2,
        "feedforward_dim": 32,
        "use_transformer": use_transformer,
    }
    model = TabularTransformer(**model_config)

    train_loader = DataLoader(
        TensorDataset(train_numeric, train_categories, y_train),
        batch_size=4096,
        shuffle=True,
        num_workers=0,
        generator=torch.Generator().manual_seed(42),
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4,
    )
    loss_fn = nn.BCEWithLogitsLoss()

    parameter_count = sum(p.numel() for p in model.parameters())
    print(f"\n{model_name}: {parameter_count:,} parameters")

    for epoch in range(1, epochs + 1):
        start = perf_counter()

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            loss_fn,
        )

        validation_scores = predict_embedding_scores(
            model,
            val_numeric,
            val_categories,
        )
        validation_ap = average_precision_score(
            y_validation.numpy(),
            validation_scores,
        )

        result = {
            "model": model_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_ap": validation_ap,
            "seconds": perf_counter() - start,
        }
        training_history.append(result)

        print(
            f"Epoch {epoch}/{epochs} | "
            f"loss={train_loss:.5f} | "
            f"validation AP={validation_ap:.4f} | "
            f"{result['seconds']:.1f}s",
            flush=True,
        )

    np.save(
        cache_dir / f"{model_name}_validation_scores.npy",
        validation_scores,
    )

    torch.save(
        {
            "model_state": model.state_dict(),
            "model_config": model_config,
            "epochs": epochs,
            "seed": 42,
            "categorical_columns": encoder.feature_names_in_.tolist(),
        },
        models_dir / f"lab07_{model_name}.pt",
    )

history = pd.DataFrame(training_history)
history.to_csv(reports_dir / "lab_07_training.csv", index=False)

history


embedding_control: 1,913 parameters
Epoch 1/5 | loss=0.00810 | validation AP=0.0228 | 7.1s
Epoch 2/5 | loss=0.00717 | validation AP=0.0351 | 6.2s
Epoch 3/5 | loss=0.00697 | validation AP=0.0419 | 6.1s
Epoch 4/5 | loss=0.00686 | validation AP=0.0582 | 6.1s
Epoch 5/5 | loss=0.00679 | validation AP=0.0598 | 6.4s

transformer: 2,785 parameters
Epoch 1/5 | loss=0.00800 | validation AP=0.0202 | 8.9s
Epoch 2/5 | loss=0.00725 | validation AP=0.0314 | 8.8s
Epoch 3/5 | loss=0.00699 | validation AP=0.0415 | 8.5s
Epoch 4/5 | loss=0.00686 | validation AP=0.0572 | 8.5s
Epoch 5/5 | loss=0.00678 | validation AP=0.0616 | 8.6s


,model,epoch,train_loss,validation_ap,seconds
0,embedding_control,1,0.008103,0.022824,7.072079
1,embedding_control,2,0.007173,0.035130,6.159238
2,embedding_control,3,0.006972,0.041920,6.105417
3,embedding_control,4,0.006858,0.058204,6.136707
4,embedding_control,5,0.006791,0.059772,6.445222
5,transformer,1,0.007999,0.020233,8.854421
6,transformer,2,0.007255,0.031424,8.813806
7,transformer,3,0.006993,0.041518,8.537056
8,transformer,4,0.006857,0.057241,8.516765
9,transformer,5,0.006777,0.061647,8.628105


In [10]:
from src.modeling import evaluate_scores

validation = pd.read_parquet(
    project_root / "data/processed/lab02_validation.parquet",
    columns=["source_row", "timestamp", "is_laundering"],
)

np.testing.assert_array_equal(
    validation["source_row"].to_numpy(),
    np.load(previous_cache / "validation_source_rows.npy"),
)
np.testing.assert_array_equal(
    validation["is_laundering"].to_numpy(),
    np.load(previous_cache / "y_validation.npy"),
)

history = pd.read_csv(
    project_root / "reports/lab_07_training.csv"
)

results = []

for name in ["embedding_control", "transformer"]:
    scores = np.load(
        cache_dir / f"{name}_validation_scores.npy"
    )

    metrics = evaluate_scores(
        validation,
        scores,
        daily_budget=50,
    )

    results.append({
        "model": name,
        **metrics,
        "run_seconds": history.loc[
            history["model"].eq(name), "seconds"
        ].sum(),
    })

comparison = pd.DataFrame(results).set_index("model")
comparison.round(4)

,average_precision,alerts,laundering_caught,precision_at_budget,recall_at_budget,run_seconds
model,,,,,,
embedding_control,0.0598,100,29,0.29,0.0280,31.9187
transformer,0.0616,100,32,0.32,0.0309,43.3502
